In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

# Set visual style
sns.set_theme(style="whitegrid")

print("==========================================================")
print("             TASK 9 LIVE VERIFICATION LOG                 ")
print("==========================================================")

# 1. Load Scaled Dataset & Edge Case Handling
df_raw = pd.read_csv('data/realistic_student_data.csv')
print(f"[VERIFIED] Raw records loaded: {len(df_raw)}")

df_raw['score'] = pd.to_numeric(df_raw['score'], errors='coerce')
df_raw['submission_date'] = pd.to_datetime(df_raw['submission_date'])

# Chronological sorting to prevent data leakage
df = df_raw.dropna(subset=['score']).sort_values('submission_date').reset_index(drop=True)
print(f"[VERIFIED] Cleaned time-series records: {len(df)}")

# 2. Train-Validation Temporal Split (Avoiding Future Data Leakage)
split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx].copy()
val = df.iloc[split_idx:].copy()

print(f"[VERIFIED] Train set: {len(train)} rows | Validation set: {len(val)} rows")

# 3. Fit Baseline & Forecast Models
# Naive Baseline (Last Value Shift)
val['naive_pred'] = val['score'].shift(1).fillna(train['score'].iloc[-1])

# Exponential Smoothing on Training set only
model = SimpleExpSmoothing(train['score']).fit(smoothing_level=0.6, optimized=False)
val['model_pred'] = model.forecast(len(val)).values

# 4. Error-Metric Calculations (MAE & RMSE)
naive_mae = np.mean(np.abs(val['score'] - val['naive_pred']))
model_mae = np.mean(np.abs(val['score'] - val['model_pred']))

print(f"[CALCULATED] Baseline Naive MAE : {naive_mae:.2f}")
print(f"[CALCULATED] Model Forecast MAE  : {model_mae:.2f}")

# 5. Confidence Interval Generation (95% CI)
future_steps = 7
future_dates = pd.date_range(start=df['submission_date'].max() + pd.Timedelta(days=1), periods=future_steps)
full_model = SimpleExpSmoothing(df['score']).fit(smoothing_level=0.6, optimized=False)
future_forecast = full_model.forecast(future_steps)

std_err = np.std(df['score'] - full_model.fittedvalues)
lower_bound = future_forecast - (1.96 * std_err)
upper_bound = future_forecast + (1.96 * std_err)

print(f"[CALCULATED] 95% Confidence Interval Band: [±{1.96 * std_err:.2f}]")

# 6. Generate Verification Plot
plt.figure(figsize=(10, 5))
plt.plot(df['submission_date'], df['score'], label='Historical Score', color='#2b5c8f', alpha=0.6)
plt.plot(val['submission_date'], val['model_pred'], label=f'Validation Forecast (MAE: {model_mae:.2f})', linestyle='--', color='#2ecc71')
plt.plot(future_dates, future_forecast, label='7-Day Future Projection', marker='o', color='#e74c3c')
plt.fill_between(future_dates, lower_bound, upper_bound, color='#e74c3c', alpha=0.2, label='95% Confidence Interval')

plt.title(f'Task 9: Verified Trend Forecast & Uncertainty Band (N={len(df)})', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Submission Date')
plt.ylabel('Score')
plt.legend()
plt.tight_layout()

# Save visual artifact
plt.savefig('trend_forecast_verification.png', dpi=300)
plt.show()

print("✅ Visual plot exported to trend_forecast_verification.png")